# LEDD — Lightweight Explainable Diffusion Detection
### Kaggle run book

Run the cells **in order**. Each phase gates the next — do not skip ahead, especially
past the leak check.

**Notebook settings (right-hand panel) before you start:**

| Setting | Value |
|---|---|
| Accelerator | `GPU T4 x2` or `P100` |
| Persistence | **Files only** (keeps `/kaggle/working` between sessions → training resumes) |
| Internet | **On** (needed to clone the repo and fetch pretrained weights) |
| Input | Add your preprocessed GenImage dataset |

Total GPU time for the core model is roughly **4–6 hours**; the full programme including
ablations and baselines is 35–50 hours, so split it across team accounts.

---
## Phase 0 — Environment and code

Expect ~2 minutes. If `cuda: False`, fix the accelerator setting before continuing.

In [ ]:
import torch, sys, platform
print('python  :', platform.python_version())
print('torch   :', torch.__version__)
print('cuda    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu     :', torch.cuda.get_device_name(0))
    print('vram GB :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Kaggle ships torch/torchvision; these three are usually missing.
!pip -q install timm fvcore thop

In [ ]:
# Public repo -> no token, no login.
import os
if not os.path.exists('/kaggle/working/ledd'):
    !git clone -q https://github.com/<your-user>/ledd.git /kaggle/working/ledd
else:
    !cd /kaggle/working/ledd && git pull -q
%cd /kaggle/working/ledd
!git log --oneline -1

### 0.1  Unit tests + smoke test

The smoke test exercises the paths most likely to break: MobileViT token-stage indexing,
AMP with attention recording, Chefer relevance shapes, band masking, deletion/insertion.

**Expect 56 tests passed and `11/11 checks passed`. Do not continue past a failure.**

In [ ]:
!python -m pytest tests/ -q

In [ ]:
!python scripts/smoke_test.py --cuda
# If anything fails:  !SMOKE_VERBOSE=1 python scripts/smoke_test.py --cuda

### 0.2  Full-size build + first paper number

Builds at 224 with real ImageNet weights and measures parameters, FLOPs and latency.
**Parameter count should land near 7M** — that is your lightweight claim.

In [ ]:
!python scripts/smoke_test.py --cuda --full-size
!python scripts/measure_efficiency.py --config configs/kaggle.yaml \
        --out /kaggle/working/runs/efficiency.json

---
## Phase 1 — Dataset validation

**This phase is the real gate.** Every number you produce later depends on it.

In [ ]:
# Point this at your dataset mount (check the right-hand Input panel for the exact path).
DATA = '/kaggle/input/genimage-224-ledd'

import os
assert os.path.isdir(DATA), f'not found: {DATA}'
print(sorted(os.listdir(DATA))[:12])

### 1.1  Archive integrity

Counts, class balance, image size, formats, duplicates across classes/generators,
and whether preprocessing settings were recorded.

In [ ]:
!python scripts/verify_archive.py --archive {DATA}

### 1.2  JPEG-history audit

A PNG/PNG archive does **not** prove the dataset is clean: PNG losslessly preserves the
JPEG artifacts of the source, and GenImage's reals are ImageNet JPEGs while several fake
subsets are PNG (Ricker et al., arXiv:2403.17608).

| Result | Action |
|---|---|
| `\|AUC−0.5\| < 0.05` | clean — proceed |
| `0.05–0.15` | mild — report it, JPEG augmentation largely covers it |
| `> 0.15` | run the equalisation cell below, then re-audit |

In [ ]:
!python scripts/audit_jpeg_history.py --archive {DATA} \
        --out /kaggle/working/jpeg_audit.json

In [ ]:
# ONLY if the audit reported a shortcut. ~30-45 min for 160k images.
# Kaggle inputs are read-only, so this writes a corrected copy to /kaggle/working.
#
# !python scripts/equalize_archive.py --src {DATA} \
#         --dst /kaggle/working/genimage_224_eq --quality 95
# DATA = '/kaggle/working/genimage_224_eq'
# !python scripts/audit_jpeg_history.py --archive {DATA}

### 1.3  Preprocessing leak check

Trains a classifier to predict **generator identity from the spectra of REAL images only**.
Real images are content-comparable across subsets, so above-chance accuracy means the
*pipeline* is discriminable and every cross-generator result would be inflated.

Chance = 1/8 = **0.125**. Non-zero exit means stop and fix.

**Save this number — it belongs in the paper. Almost no detection paper reports it.**

In [ ]:
!python scripts/run_leak_check.py --archive {DATA} \
        --generators sd_v14 sd_v15 wukong vqdm biggan glide adm midjourney \
        --out /kaggle/working/runs/leak_check.json

### 1.4  Generator split (confirm before training)

| Role | Generators | Why |
|---|---|---|
| Train | sd_v14, sd_v15, wukong, vqdm, biggan, glide | SD family + diverse architectures |
| Validation | **adm** | moderately dissimilar — checkpoint selection and tuning **only** |
| Test | **midjourney** | hardest transfer target per GenImage's correlation analysis |

**Do not look at Midjourney numbers until the very end.** That is the entire point of the
three-role split, and a reviewer will ask.

In [ ]:
!python -c "from ledd.utils.config import load_config; c=load_config('configs/default.yaml')['data']; print(c['train_generators']); print('val :', c['val_generator']); print('test:', c['test_generators'])"

---
## Phase 2 — Training

Checkpoints go to `/kaggle/working/runs/...`. With **Persistence: Files only**, a killed
session resumes automatically — just re-run the same cell.

Run this in a second cell during the first epoch to check you are not dataloader-bound:
`!nvidia-smi --query-gpu=utilization.gpu --format=csv -l 5`

If GPU utilisation stays below ~60%: raise `data.num_workers` to 8, or drop
`augment.jpeg_prob` to 0.3. Judge from epoch 2 — the first is always slow (cold filesystem).

### 2.1  Stage 2 — frequency stream first  (~30–60 min)

Under 1M parameters, minutes per epoch. Train the cheap stream first because it tells you
immediately whether the frequency signal is real.

**Expect validation-generator AUC around 0.65–0.85.** Not 0.99 — diffusion images lack the
obvious GAN grid artifacts. If it sits at 0.5, stop and diagnose the dataloader labels.

In [ ]:
!python scripts/train.py --config configs/kaggle.yaml \
    --set stage=frequency data.root={DATA} train.epochs=15 train.batch_size=256 \
         loss.supcon_weight=0.0 loss.band_entropy_weight=0.0 \
         train.ckpt_dir=/kaggle/working/runs/stage2_frequency

In [ ]:
import pandas as pd
pd.read_csv('/kaggle/working/runs/stage2_frequency/metrics.csv')

### 2.2  Which bands did it learn?

**Watch for mid-band concentration.** FIRE (CVPR 2025) found the generalisable signal lives
in the mid-band using reconstruction error. If your attention independently agrees via a
completely different mechanism, that is a genuine result. If it disagrees, that is also a
result — either way it earns a paragraph.

In [ ]:
!python scripts/run_explainability.py --config configs/kaggle.yaml \
    --set data.root={DATA} train.ckpt_dir=/kaggle/working/runs/stage2_frequency \
    --ckpt /kaggle/working/runs/stage2_frequency/best.pth --n-batches 4 \
    --out /kaggle/working/runs/explain_stage2

### 2.3  Stage 1 — spatial stream  (~1.5–2.5 h)

**Expect higher in-distribution AUC than Stage 2, but a *larger gap* between
in-distribution and validation-generator AUC.** The spatial stream overfits to generator
fingerprints — precisely the weakness the frequency stream and contrastive loss exist to
offset. Record both numbers; that gap is a figure in your paper.

In [ ]:
!python scripts/train.py --config configs/kaggle.yaml \
    --set stage=spatial data.root={DATA} train.epochs=12 train.batch_size=96 \
         train.lr_backbone=1e-4 loss.supcon_weight=0.0 loss.band_entropy_weight=0.0 \
         train.ckpt_dir=/kaggle/working/runs/stage1_spatial

### 2.4  Stage 3 — joint  (~1.5–2.5 h)

Full loss (BCE + SupCon + band entropy), modality dropout, cross-attention fusion.

**The number that matters: validation-generator AUC versus both single-stream runs.**
If joint does not beat both, the fusion is not earning its parameters — diagnose before
running any ablations.

In [ ]:
!python scripts/train.py --config configs/kaggle.yaml \
    --set data.root={DATA} \
         init.spatial_ckpt=/kaggle/working/runs/stage1_spatial/best.pth \
         init.frequency_ckpt=/kaggle/working/runs/stage2_frequency/best.pth \
         train.ckpt_dir=/kaggle/working/runs/stage3_joint

In [ ]:
import pandas as pd, json
for name in ['stage1_spatial','stage2_frequency','stage3_joint']:
    df = pd.read_csv(f'/kaggle/working/runs/{name}/metrics.csv')
    best = df['val_gen_auc'].max()
    ind  = df.loc[df['val_gen_auc'].idxmax(), 'indist_auc']
    print(f'{name:18s}  val-generator AUC {best:.4f}   in-distribution AUC {ind:.4f}   gap {ind-best:+.4f}')

---
## Phase 3 — Evaluation, explanation, efficiency

### 3.1  Full protocol table

In-distribution, both cross-generator splits, and every degradation.
Add `--ood-root` once your SDXL/SD3/Flux set exists.

In [ ]:
!python scripts/evaluate.py --config configs/kaggle.yaml \
    --set data.root={DATA} \
    --ckpt /kaggle/working/runs/stage3_joint/best.pth \
    --splits /kaggle/working/runs/stage3_joint/splits.json \
    --out /kaggle/working/runs/protocol.json

### 3.2  Faithfulness

**The maps must beat the random-saliency control**, or they are decoration, not explanation.
Deletion: lower is better. Insertion: higher is better.

Also check `balance_validation` — if the claimed spatial-vs-frequency split does not
correlate with causal stream deletion, report only the causal version.

In [ ]:
!python scripts/run_explainability.py --config configs/kaggle.yaml \
    --set data.root={DATA} train.ckpt_dir=/kaggle/working/runs/stage3_joint \
    --ckpt /kaggle/working/runs/stage3_joint/best.pth --n-batches 8 \
    --out /kaggle/working/runs/explain_final

import json
print(json.dumps(json.load(open('/kaggle/working/runs/explain_final/faithfulness.json')), indent=2))

### 3.3  Efficiency (final model)

In [ ]:
!python scripts/measure_efficiency.py --config configs/kaggle.yaml \
    --ckpt /kaggle/working/runs/stage3_joint/best.pth \
    --out /kaggle/working/runs/efficiency_final.json

---
## Phase 4 — Ablations

**Run the concat ablation first.** Everything rests on cross-attention beating
concatenation — DIRE found naive fusion actively hurt, and a null result here changes the
paper. Use fewer epochs for ablations; you are measuring a delta, not chasing peak accuracy.

In [ ]:
ABL = ('data.root={} '
       'init.spatial_ckpt=/kaggle/working/runs/stage1_spatial/best.pth '
       'init.frequency_ckpt=/kaggle/working/runs/stage2_frequency/best.pth '
       'train.epochs=6').format(DATA)
print(ABL)

In [ ]:
# A1 — the mandatory one: cross-attention vs concatenation
!python scripts/train.py --config configs/ablations/fusion_concat.yaml \
    --set {ABL} train.ckpt_dir=/kaggle/working/runs/abl_concat

In [ ]:
# A2 — without the contrastive loss
!python scripts/train.py --config configs/ablations/no_supcon.yaml \
    --set {ABL} train.ckpt_dir=/kaggle/working/runs/abl_no_supcon

# A3 — without the band-entropy regulariser
!python scripts/train.py --config configs/ablations/no_band_entropy.yaml \
    --set {ABL} train.ckpt_dir=/kaggle/working/runs/abl_no_entropy

# A4 — without modality dropout
!python scripts/train.py --config configs/ablations/no_modality_dropout.yaml \
    --set {ABL} train.ckpt_dir=/kaggle/working/runs/abl_no_moddrop

In [ ]:
# A5 — frequency representation (a locked paper contribution)
!python scripts/train.py --config configs/kaggle.yaml \
    --set stage=frequency data.root={DATA} train.epochs=10 train.batch_size=256 \
         model.frequency.representation=magnitude_phase \
         train.ckpt_dir=/kaggle/working/runs/abl_phase

!python scripts/train.py --config configs/kaggle.yaml \
    --set stage=frequency data.root={DATA} train.epochs=10 train.batch_size=256 \
         model.frequency.representation=dct \
         train.ckpt_dir=/kaggle/working/runs/abl_dct

In [ ]:
# A6 — number of frequency bands
for n in [8, 16]:
    !python scripts/train.py --config configs/kaggle.yaml \
        --set stage=frequency data.root={DATA} train.epochs=10 train.batch_size=256 \
             model.frequency.n_bands={n} \
             train.ckpt_dir=/kaggle/working/runs/abl_bands{n}

---
## Phase 5 — Baselines

Same splits, same preprocessing, same degradations — otherwise the comparison is worthless.

**Schedule FIRE first.** It runs an LDM autoencoder per image and costs 6–10 h, more than
your entire model pipeline. A surprise there should not eat your model phase.

In [ ]:
# NPR — 1.44M params, the lightweight-pillar comparison (~45 min)
!python scripts/train_baseline.py --config configs/baselines/npr.yaml \
    --set data.root={DATA} train.ckpt_dir=/kaggle/working/runs/base_npr

In [ ]:
# CNNDetection — their augmentation recipe (~1.5-2 h)
!python scripts/train_baseline.py --config configs/baselines/cnndetection.yaml \
    --set data.root={DATA} train.ckpt_dir=/kaggle/working/runs/base_cnndetection

# Plain ResNet50 anchor (~1.5-2 h)
!python scripts/train_baseline.py --config configs/baselines/resnet50.yaml \
    --set data.root={DATA} train.ckpt_dir=/kaggle/working/runs/base_resnet50

**FIRE** (Chu et al., CVPR 2025) runs from the authors' own repository —
see `ledd/baselines/README.md`. Clone into `/kaggle/working/third_party/FIRE`, fetch their
LDM autoencoder weights, train on a reduced subset if compute is tight (**record the subset
size in the paper**), then score it through `scripts/evaluate.py --external fire`.

**DIRE** is inference-only on the test sets — per-image DDIM inversion makes full training
infeasible on free-tier GPUs.

---
## Phase 6 — Collect results

Pulls every run into one table for the paper.

In [ ]:
import glob, os, json
import pandas as pd

rows = []
for path in sorted(glob.glob('/kaggle/working/runs/*/metrics.csv')):
    name = os.path.basename(os.path.dirname(path))
    df = pd.read_csv(path)
    if 'val_gen_auc' not in df or df['val_gen_auc'].isna().all():
        continue
    i = df['val_gen_auc'].idxmax()
    rows.append({'run': name,
                 'epochs': len(df),
                 'val_gen_auc': round(df.loc[i, 'val_gen_auc'], 4),
                 'val_gen_f1': round(df.loc[i, 'val_gen_f1'], 4),
                 'indist_auc': round(df.loc[i, 'indist_auc'], 4),
                 'gap': round(df.loc[i, 'indist_auc'] - df.loc[i, 'val_gen_auc'], 4),
                 'mins': round(df['secs'].sum()/60, 1)})

summary = pd.DataFrame(rows).sort_values('val_gen_auc', ascending=False)
summary.to_csv('/kaggle/working/runs/summary.csv', index=False)
summary

In [ ]:
# Per-generator breakdown from the protocol run
import json, pandas as pd
p = json.load(open('/kaggle/working/runs/protocol.json'))
for split in ['in_distribution', 'cross_generator_val', 'cross_generator_test']:
    if split in p:
        print(f"\n{split}: AUC {p[split].get('auc'):.4f}  F1 {p[split].get('f1'):.4f}")
        pg = p[split].get('per_generator', {})
        if pg:
            print(pd.DataFrame(pg).T[['auc','f1','n']].round(4).to_string())

print('\nRobustness:')
print(pd.DataFrame({k: {'auc': v.get('auc'), 'f1': v.get('f1')}
                    for k, v in p.get('robustness', {}).items()}).T.round(4).to_string())

---
## Saving work between sessions

With **Persistence: Files only**, `/kaggle/working` survives. For safety across quota
resets, publish the checkpoints as a dataset and attach it as an input next session:

```python
!kaggle datasets init -p /kaggle/working/runs
# edit dataset-metadata.json, then:
!kaggle datasets create -p /kaggle/working/runs --dir-mode zip
```

### Order, at a glance

```
env → tests → smoke → efficiency
  → verify_archive → jpeg_audit → leak_check
    → Stage 2 → band inspection → Stage 1 → Stage 3
      → protocol → faithfulness → efficiency
        → concat ablation → remaining ablations
          → NPR → CNNDetection → ResNet50 → FIRE
            → summary
```